# Odisha Crop Advisory: crop classifier (RF + XGBoost) and yield ranges
**No values in the datasets are added, changed or removed** (the yield file is only filtered to Odisha). All 22 recommender crops and all 43 Odisha crops are kept.

Yield is shown as a **typical range taken from the data**, not a model prediction, because about half of the source yield values are in a different unit (see step 5).

Put both CSVs next to this notebook and run top to bottom.

In [ ]:
import numpy as np, pandas as pd, joblib, os
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, r2_score, mean_absolute_error
from xgboost import XGBClassifier, XGBRegressor   # pip install xgboost

# ---- files (edit names if yours differ) ----
YIELD_CSV = "Indian_crop_production_yield_dataset.csv"
REC_CSV   = "Crop_recommendation.csv"

# ---- settings ----
STATE = "Odisha"
FILTER_UNIT_OUTLIERS = False  # False = keep every row. True would drop rows with yield far above the crop's normal range
UNIT_K = 5                    # keep rows with yield <= UNIT_K x the crop's 25th percentile
MIN_ROWS_PER_CROP = 1         # 1 = keep every crop that has at least one row
SPLIT_YEAR = 2012             # yield model: train <= this year, test after (time-based)

os.makedirs("data/processed", exist_ok=True); os.makedirs("models", exist_ok=True)

## 1. Yield dataset: filter Odisha and clean rows

In [ ]:
raw = pd.read_csv(YIELD_CSV)
y = raw[raw["State_Name"] == STATE].copy()
report = {"odisha rows": len(y)}

# missing / zero values -> remove row
y = y.dropna(subset=["Area", "Production", "yield"])
y = y[(y["Area"] > 0) & (y["Production"] > 0) & (y["yield"] > 0)]
report["after missing/zero removal"] = len(y)

# rows whose yield is in a different unit from the rest of that crop (e.g. all of 2016-2019) -> remove row
if FILTER_UNIT_OUTLIERS:
    q25 = y.groupby("Crop")["yield"].transform(lambda s: s.quantile(0.25))
    y = y[y["yield"] <= UNIT_K * q25]
    report["after unit-consistency filter"] = len(y)

# crops with too little data -> remove
counts = y["Crop"].value_counts()
y = y[y["Crop"].isin(counts[counts >= MIN_ROWS_PER_CROP].index)]
report["after min-rows-per-crop"] = len(y)

print(report)
print("districts:", y["District_Name"].nunique(), "| crops:", y["Crop"].nunique(), "| years:", y["Crop_Year"].min(), "-", y["Crop_Year"].max())
print(y.groupby("Crop_Year").size().to_dict())
y.to_csv("data/processed/odisha_yield_clean.csv", index=False)

## 2. Check: median yield by year after cleaning

In [ ]:
crops = [c for c in ["Rice", "Maize", "Groundnut", "Potato", "Sugarcane"] if c in y["Crop"].unique()]
piv = y[y["Crop"].isin(crops)].pivot_table(index="Crop_Year", columns="Crop", values="yield", aggfunc="median")
piv.plot(figsize=(10, 4), logy=True, title="Median yield by year (log scale), after cleaning"); plt.show()

## 3. Crop recommendation dataset: keep ALL 22 crops
No rows are removed. The mapping only links a recommender label to its crop name in the Odisha yield data, so yield can be predicted where Odisha yield data exists.

In [ ]:
rec = pd.read_csv(REC_CSV).dropna()          # all 22 crops kept
LABEL_TO_YIELD_CROP = {"rice": "Rice", "maize": "Maize", "chickpea": "Gram", "pigeonpeas": "Arhar/Tur",
                       "mungbean": "Moong(Green Gram)", "blackgram": "Urad", "lentil": "Masoor",
                       "cotton": "Cotton(lint)", "jute": "Jute"}
yield_crops = set(y["Crop"].unique())
label_to_yield_crop = {l: c for l, c in LABEL_TO_YIELD_CROP.items() if c in yield_crops}
print("crops in recommender:", rec["label"].nunique(), "| rows:", len(rec))
print("crops with Odisha yield data:", label_to_yield_crop)
print("crops WITHOUT Odisha yield data:", sorted(set(rec["label"]) - set(label_to_yield_crop)))
rec.to_csv("data/processed/odisha_crop_rec_clean.csv", index=False)

## 4. Crop classifier: Random Forest vs XGBoost

In [ ]:
FEAT = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
le = LabelEncoder().fit(rec["label"])
Xtr, Xte, ytr, yte = train_test_split(rec[FEAT], le.transform(rec["label"]),
                                      test_size=0.2, stratify=rec["label"], random_state=42)
cands = {"rf": RandomForestClassifier(n_estimators=300, random_state=42),
         "xgb": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, random_state=42)}
scores = {}
for k, m in cands.items():
    m.fit(Xtr, ytr); pred = m.predict(Xte); scores[k] = accuracy_score(yte, pred)
    print(k, "accuracy:", round(scores[k], 4))
best = max(scores, key=scores.get); print("best:", best)
print(classification_report(yte, cands[best].predict(Xte), target_names=le.classes_))
joblib.dump({"model": cands[best], "label_encoder": le, "features": FEAT, "name": best,
             "label_to_yield_crop": label_to_yield_crop}, "models/crop_model.pkl")

## 5. Yield ranges per crop and district (saved as the second .pkl)
About half of the source yield values are in a different unit (for example rice at 150+ t/ha), which breaks any yield model. So the app shows a typical range instead. When *summarising*, rows with yield above `UNIT_K` x the crop's 25th percentile are left out of the summary only. The dataset itself is not modified.

In [ ]:
q25 = y.groupby("Crop")["yield"].transform(lambda s: s.quantile(0.25))
consistent = y[y["yield"] <= UNIT_K * q25]     # used for summaries only; y is untouched

def summarise(g, total):
    return {"n": int(len(g)), "rows_total": int(total), "p25": float(g["yield"].quantile(.25)),
            "median": float(g["yield"].median()), "p75": float(g["yield"].quantile(.75)),
            "years": (int(g["Crop_Year"].min()), int(g["Crop_Year"].max()))}

by_crop, by_crop_district = {}, {}
for crop, g in consistent.groupby("Crop"):
    by_crop[crop] = summarise(g, (y["Crop"] == crop).sum())
    by_crop_district[crop] = {d: summarise(gd, ((y["Crop"] == crop) & (y["District_Name"] == d)).sum())
                              for d, gd in g.groupby("District_Name")}

joblib.dump({"by_crop": by_crop, "by_crop_district": by_crop_district,
             "districts": sorted(y["District_Name"].unique()), "unit": "tonnes per hectare",
             "note": "Typical range (25th-75th percentile) from Odisha rows with consistent units."},
            "models/yield_stats.pkl")
print("crops with yield ranges:", len(by_crop))
print(pd.DataFrame(by_crop).T[["n", "rows_total", "p25", "median", "p75"]].round(2).sort_values("n", ascending=False).head(12))

## 6. Sanity test: load the saved files the way `app.py` will

In [ ]:
cm = joblib.load("models/crop_model.pkl"); ys = joblib.load("models/yield_stats.pkl")
row = pd.DataFrame([[90, 42, 43, 21, 82, 6.5, 200]], columns=cm["features"])
proba = pd.Series(cm["model"].predict_proba(row)[0], index=cm["label_encoder"].classes_).sort_values(ascending=False)
print("top 3 crops:", proba.head(3).round(3).to_dict())
crop = cm["label_to_yield_crop"].get("rice")
print("rice in Ganjam:", ys["by_crop_district"][crop].get("GANJAM"))
print("rice, all Odisha:", ys["by_crop"][crop])